In [1]:
from fuzzycar import Car
from pynq import Overlay , MMIO

In [2]:
ol = Overlay("car.bit", ignore_version=True)

In [3]:
car = Car(ol)

Configuring SPI controller...


In [4]:
gpio = MMIO(ol.axi_gpio_0.mmio.base_addr, 0x10000 , debug=False)

In [5]:
gpio.write(0x00,0xff)

In [6]:
car.pasback.read_distance()

9

In [7]:
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl
import time
from IPython.display import clear_output

# Define fuzzy variables
driver_sensor = ctrl.Antecedent(np.arange(6, 256, 1), 'driver_sensor')
front_sensor = ctrl.Antecedent(np.arange(6, 256, 1), 'front_sensor')
passenger_sensor = ctrl.Antecedent(np.arange(6, 256, 1), 'passenger_sensor')
rear_driver_sensor = ctrl.Antecedent(np.arange(6, 256, 1), 'rear_driver_sensor')
rear_passenger_sensor = ctrl.Antecedent(np.arange(6, 256, 1), 'rear_passenger_sensor')
turning_degree = ctrl.Consequent(np.arange(600, 1001, 1), 'turning_degree')

# Membership functions for driver_sensor
driver_sensor['Close'] = fuzz.trapmf(driver_sensor.universe, [6, 6, 55, 65])
driver_sensor['Medium'] = fuzz.trapmf(driver_sensor.universe, [55, 65, 75, 85])
driver_sensor['Far'] = fuzz.trapmf(driver_sensor.universe, [75, 85, 255, 255])

# Membership functions for front_sensor
front_sensor['Close'] = fuzz.trapmf(front_sensor.universe, [6, 6, 45, 55])
front_sensor['Medium'] = fuzz.trapmf(front_sensor.universe, [45, 55, 65, 75])
front_sensor['Far'] = fuzz.trapmf(front_sensor.universe, [65, 75, 255, 255])

# Membership functions for passenger_sensor
passenger_sensor['Close'] = fuzz.trapmf(passenger_sensor.universe, [6, 6, 55, 65])
passenger_sensor['Medium'] = fuzz.trapmf(passenger_sensor.universe, [55, 65, 75, 85])
passenger_sensor['Far'] = fuzz.trapmf(passenger_sensor.universe, [75, 85, 255, 255])

# Membership functions for rear_driver_sensor
rear_driver_sensor['Close'] = fuzz.trapmf(rear_driver_sensor.universe, [6, 6, 55, 65])
rear_driver_sensor['Medium'] = fuzz.trapmf(rear_driver_sensor.universe, [55, 65, 75, 85])
rear_driver_sensor['Far'] = fuzz.trapmf(rear_driver_sensor.universe, [75, 85, 255, 255])

# Membership functions for rear_passenger_sensor
rear_passenger_sensor['Close'] = fuzz.trapmf(rear_passenger_sensor.universe, [6, 6, 55, 65])
rear_passenger_sensor['Medium'] = fuzz.trapmf(rear_passenger_sensor.universe, [55, 65, 75, 85])
rear_passenger_sensor['Far'] = fuzz.trapmf(rear_passenger_sensor.universe, [75, 85, 255, 255])

# Membership functions for Turning Degree
turning_degree['Hard Left'] = fuzz.trimf(turning_degree.universe, [600, 600, 700])
turning_degree['Soft Left'] = fuzz.trimf(turning_degree.universe, [600, 700, 800])
turning_degree['Straight'] = fuzz.trimf(turning_degree.universe, [700, 800, 900])
turning_degree['Soft Right'] = fuzz.trimf(turning_degree.universe, [800, 900, 1000])
turning_degree['Hard Right'] = fuzz.trimf(turning_degree.universe, [900, 1000, 1000])

# Define rules
rule1 = ctrl.Rule(driver_sensor['Close'] & front_sensor['Medium'] & ~passenger_sensor['Close'], turning_degree['Soft Right'])
rule2 = ctrl.Rule(driver_sensor['Close'] & ~front_sensor['Close'] & ~passenger_sensor['Close'], turning_degree['Hard Right'])
rule3 = ctrl.Rule(~driver_sensor['Close'] & front_sensor['Medium'] & ~passenger_sensor['Far'], turning_degree['Soft Left'])
rule4 = ctrl.Rule(~driver_sensor['Close'] & front_sensor['Close'] & ~passenger_sensor['Far'], turning_degree['Hard Left'])
rule5 = ctrl.Rule(driver_sensor['Close'] & front_sensor['Far'] & ~passenger_sensor['Close'], turning_degree['Straight'])
rule6 = ctrl.Rule(~driver_sensor['Close'] & front_sensor['Far'] & passenger_sensor['Close'], turning_degree['Straight'])
rule7 = ctrl.Rule(~driver_sensor['Close'] & front_sensor['Far'] & ~passenger_sensor['Close'], turning_degree['Straight'])
rule8 = ctrl.Rule(~driver_sensor['Close'] & front_sensor['Medium'] & passenger_sensor['Far'], turning_degree['Soft Right'])
#rule9 = ctrl.Rule(front_sensor['Close'] & passenger_sensor['Far'], turning_degree['Hard Right'])
rule9 = ctrl.Rule(~driver_sensor['Close'] & front_sensor['Close'] & passenger_sensor['Far'], turning_degree['Hard Right'])
rule10 = ctrl.Rule(driver_sensor['Close'] & front_sensor['Close'] & passenger_sensor['Close'], turning_degree['Straight'])
rule11 = ctrl.Rule(driver_sensor['Medium'] & front_sensor['Medium'] & passenger_sensor['Medium'], turning_degree['Soft Right'])
rule12 = ctrl.Rule(driver_sensor['Medium'] & front_sensor['Close'] & passenger_sensor['Close'], turning_degree['Hard Left'])
rule13 = ctrl.Rule(driver_sensor['Far'] & front_sensor['Close'] & passenger_sensor['Close'], turning_degree['Hard Left'])
rule14 = ctrl.Rule(driver_sensor['Close'] & front_sensor['Close'] & ~passenger_sensor['Close'], turning_degree['Hard Right'])

# New rules for the back sensors
rule15 = ctrl.Rule(rear_driver_sensor['Close'] & rear_passenger_sensor['Close'], turning_degree['Straight'])  # reverse alert
rule16 = ctrl.Rule(rear_driver_sensor['Close'] & ~rear_passenger_sensor['Close'] & front_sensor['Medium'], turning_degree['Soft Right'])
rule17 = ctrl.Rule(rear_passenger_sensor['Close'] & ~rear_driver_sensor['Close'] & front_sensor['Medium'], turning_degree['Soft Left'])
rule18 = ctrl.Rule(rear_driver_sensor['Close'] & rear_passenger_sensor['Far'] & driver_sensor['Far'], turning_degree['Hard Right'])
rule19 = ctrl.Rule(rear_passenger_sensor['Close'] & rear_driver_sensor['Far'] & driver_sensor['Far'], turning_degree['Hard Left'])

# Create control system and simulation
turning_ctrl = ctrl.ControlSystem([rule1, rule2, rule3, rule4, rule5, rule6, rule7, rule8, rule9, rule10, rule11, rule12, rule13, rule14, rule15, rule16, rule17, rule18, rule19])
turning_simulation = ctrl.ControlSystemSimulation(turning_ctrl)


In [8]:
# Just change this value and run the cell again
control_command = 'F'  # Set to 'S' to idle, 'F' to move forward
car.steering.set_pwm_time(50, 1500)
car.motor.set_pwm_time(50, 1600)


In [ ]:
from IPython.display import clear_output
from time import sleep

last_valid_pwm = 800  # default PWM

try:
    while True:
        # Check value from other cell
        command = control_command.upper()

        if command == 'S':
            car.motor.set_pwm_time(50, 1600)
            car.steering.set_pwm_time(50, 800)
            print("Motor is idle")

        elif command == 'F':
            # Read sensors
            gpio.write(0x00, 0x06)
            driver_distance = car.driver.read_distance()
            passenger_distance = car.passenger.read_distance()
            front_distance = car.front.read_distance()
            gpio.write(0x00, 0x08)
            pasfront_distance = car.pasfront.read_distance()
            gpio.write(0x00, 0x10)
            drifront_distance = car.drifront.read_distance()

            gpio.write(0x00, 0x20)
            sleep(0.01)
            try:
                pasback_distance = car.pasback.read_distance()
                print("✔ pasback sensor read successfully.")
            except Exception as e:
                print(f"❌ Error reading pasback sensor: {e}")

            gpio.write(0x00, 0x40) 
            sleep(0.01) 
            try:
                drivback_distance = car.drivback.read_distance()
                print("✔ drivback sensor read successfully.")
            except Exception as e:
                print(f"❌ Error reading drivback sensor: {e}")

            gpio.write(0x00, 0x00)

            print("Sensor Readings:")
            print(f"  Driver:       {driver_distance}")
            print(f"  Passenger:    {passenger_distance}")
            print(f"  Front:        {front_distance}")
            print(f"  Pass Front:   {pasfront_distance}")
            print(f"  Drive Front:  {drifront_distance}")
            print(f"  Pass Back:    {pasback_distance}")
            print(f"  Drive Back:   {drivback_distance}")

            # Fuzzy logic
            turning_simulation.input['driver_sensor'] = passenger_distance
            turning_simulation.input['front_sensor'] = front_distance
            turning_simulation.input['passenger_sensor'] = driver_distance
            turning_simulation.input['rear_driver_sensor'] = drivback_distance
            turning_simulation.input['rear_passenger_sensor'] = pasback_distance
            turning_simulation.compute()

            pwm = turning_simulation.output.get('turning_degree')

            if pwm is not None:
                last_valid_pwm = pwm
                car.steering.set_pwm_time(50, pwm)
            else:
                print("⚠️  No valid steering output — using last good PWM value.")
                car.steering.set_pwm_time(50, last_valid_pwm)
                
#------------------ NEW: rear safety + boxed-in logic (numbers only) -------------------------------------------------------
            try:
                rear_too_close = (pasback_distance < 8) or (drivback_distance < 8)
            except:
                rear_too_close = False

            # Both front and back too close then stop and determine direction to turn
            if front_distance < 12 and rear_too_close:
                print("🟥 Boxed in")
                try:
                    # If passenger front is tighter, steer right; else steer left
                    if pasfront_distance < drifront_distance:
                        car.steering.set_pwm_time(50, 1000)
                    else:
                        car.steering.set_pwm_time(50, 600)
                except:
                    car.steering.set_pwm_time(50, 800)
                car.motor.set_pwm_time(50, 1600)
                sleep(0.40)
                clear_output(wait=True)
                sleep(0.05)
                continue

            # back sensor only close 
            if rear_too_close and not front_distance < 8:
                print("🔙 Rear obstacle too close — stopping and moving forward to clear.")
                car.motor.set_pwm_time(50, 1600)
                try:
                    if pasback_distance is None and drivback_distance is None:
                        car.steering.set_pwm_time(50, 800)
                    elif pasback_distance is None:
                        car.steering.set_pwm_time(50, 600)   # assume driver rear closer
                    elif drivback_distance is None:
                        car.steering.set_pwm_time(50, 1000)  # assume passenger rear closer
                    elif pasback_distance < drivback_distance:
                        car.steering.set_pwm_time(50, 1000)  # passenger rear closer → turn right
                    else:
                        car.steering.set_pwm_time(50, 600)   # driver rear closer → turn left
                except:
                    car.steering.set_pwm_time(50, 800)

                car.motor.set_pwm_time(50, 1640)
                sleep(0.3)
                car.motor.set_pwm_time(50, 1600)
                clear_output(wait=True)
                sleep(0.05)
                continue

            if front_distance < 8:
                print("🚨 Obstacle detected — reversing.")
                car.steering.set_pwm_time(50, 600 if pasfront_distance < drifront_distance else 1000)
                car.motor.set_pwm_time(50, 1379)
                sleep(0.1)
                car.motor.set_pwm_time(50, 1600)
                sleep(0.1)
                car.motor.set_pwm_time(50, 1379)
                sleep(0.2)
            else:
                car.motor.set_pwm_time(50, 1690)

        clear_output(wait=True)
        sleep(0.05)

except KeyboardInterrupt:
    car.steering.set_pwm_time(50, 800)
    car.motor.set_pwm_time(50, 1600)
    print("Stopped by user.")
    #WIRED

✔ pasback sensor read successfully.
✔ drivback sensor read successfully.
Sensor Readings:
  Driver:       7
  Passenger:    22
  Front:        17
  Pass Front:   18
  Drive Front:  15
  Pass Back:    6
  Drive Back:   7
🔙 Rear obstacle too close — stopping and moving forward to clear.


In [ ]:
# WIRELESS MODE

In [ ]:
import time
from time import sleep
from IPython.display import clear_output

# Initialize last known good PWM value
last_valid_pwm = 800  # or your desired default center PWM

# MAIN LOOP
while True:
    we = car.lora.transfer([0x00])
    wee = car.lora.transfer([0x00])
    received_chars = set(chr(byte[0]) for byte in (we, wee) if byte and byte[0] not in (0, 255))
    
    if 'S' in received_chars:
        car.motor.set_pwm_time(50, 1600)
        car.steering.set_pwm_time(50, 800)
        print("Motor is idle")
        
    elif 'F' in received_chars:
        # Read sensor values
        gpio.write(0x00, 0x01)
        driver_distance = car.driver.read_distance()
        gpio.write(0x00, 0x02)
        passenger_distance = car.passenger.read_distance()
        gpio.write(0x00, 0x04)
        front_distance = car.front.read_distance()
        gpio.write(0x00, 0x08)
        pasfront_distance = car.pasfront.read_distance()
        gpio.write(0x00, 0x10)
        drifront_distance = car.drifront.read_distance()
        gpio.write(0x00, 0x00)

        print("Sensor Readings:")
        print(f"  Driver:       {driver_distance}")
        print(f"  Passenger:    {passenger_distance}")
        print(f"  Front:        {front_distance}")
        print(f"  Pass Front:   {pasfront_distance}")
        print(f"  Drive Front:  {drifront_distance}")

        # Set fuzzy logic inputs
        turning_simulation.input['driver_sensor'] = passenger_distance
        turning_simulation.input['front_sensor'] = front_distance
        turning_simulation.input['passenger_sensor'] = driver_distance

        # Compute fuzzy logic result
        turning_simulation.compute()
        pwm = turning_simulation.output.get('turning_degree')

        if pwm is not None:
            last_valid_pwm = pwm
            car.steering.set_pwm_time(50, pwm)
        else:
            print("⚠️  No valid steering output — using last good PWM value.")
            car.steering.set_pwm_time(50, last_valid_pwm)

        if front_distance < 9:
            print("🚨 Obstacle detected — reversing.")
            car.steering.set_pwm_time(50, 1000 if pasfront_distance < drifront_distance else 600)
            car.motor.set_pwm_time(50, 1370)
            sleep(0.1)
            car.motor.set_pwm_time(50, 1600)
            sleep(0.1)
            car.motor.set_pwm_time(50, 1370)
            sleep(0.5)
        else:
            car.motor.set_pwm_time(50, 1680)

    clear_output(wait=True)


In [ ]:
from IPython.display import clear_output
import time

last_valid_pwm = 800  # Default center PWM

try:
    while True:
        # Select each sensor with GPIO and read distances
        gpio.write(0x00, 0x01)
        driver_distance = car.driver.read_distance()

        gpio.write(0x00, 0x02)
        passenger_distance = car.passenger.read_distance()

        gpio.write(0x00, 0x04)
        front_distance = car.front.read_distance()

        gpio.write(0x00, 0x08)
        pasfront_distance = car.pasfront.read_distance()

        gpio.write(0x00, 0x10)
        drifront_distance = car.drifront.read_distance()

        gpio.write(0x00, 0x00)  # Deselect all sensors

        # Print current sensor readings
        print("Sensor Readings:")
        print(f"  Driver:       {driver_distance}")
        print(f"  Passenger:    {passenger_distance}")
        print(f"  Front:        {front_distance}")
        print(f"  Pass Front:   {pasfront_distance}")
        print(f"  Drive Front:  {drifront_distance}")

        # Set fuzzy logic inputs
        turning_simulation.input['driver_sensor'] = passenger_distance
        turning_simulation.input['front_sensor'] = front_distance
        turning_simulation.input['passenger_sensor'] = driver_distance

        # Compute fuzzy logic result
        turning_simulation.compute()
        pwm = turning_simulation.output.get('turning_degree')

        # Apply PWM if valid, else use last good value
        if pwm is not None:
            last_valid_pwm = pwm
            car.steering.set_pwm_time(50, pwm)
        else:
            print("⚠️  No valid steering output — using last good PWM value.")
            car.steering.set_pwm_time(50, last_valid_pwm)

        # Obstacle detected — reverse
        if front_distance < 0:
            print("🚨 Obstacle detected — reversing.")
            reverse_pwm = 1000 if pasfront_distance < drifront_distance else 600
            car.steering.set_pwm_time(50, reverse_pwm)
            car.motor.set_pwm_time(50, 1600)
            time.sleep(0.1)
            car.motor.set_pwm_time(50, 1600)
            time.sleep(0.1)
            car.motor.set_pwm_time(50, 1600)
            time.sleep(1)
        else:
            car.motor.set_pwm_time(50, 1600)

        # Short delay and clear output for next loop
        time.sleep(0.001)
        clear_output(wait=True)

except KeyboardInterrupt:
    car.steering.set_pwm_time(50, 800)
    car.motor.set_pwm_time(50, 0)
    print("Stopped by user.")


In [10]:
car.motor.set_pwm_time(50,1600)
car.steering.set_pwm_time(50,800)

In [11]:
driver_distance = car.passenger.read_distance()

In [ ]:
from IPython.display import clear_output
from time import sleep

last_valid_pwm = 800  # default PWM

try:
    while True:
        # Check value from other cell
        command = control_command.upper()

        if command == 'S':
            car.motor.set_pwm_time(50, 1600)
            car.steering.set_pwm_time(50, 800)
            print("Motor is idle")

        elif command == 'F':
            # Read sensors
            gpio.write(0x00, 0x06)
            driver_distance = car.driver.read_distance()
            passenger_distance = car.passenger.read_distance()
            front_distance = car.front.read_distance()
            gpio.write(0x00, 0x08)
            pasfront_distance = car.pasfront.read_distance()
            gpio.write(0x00, 0x10)
            drifront_distance = car.drifront.read_distance()

            gpio.write(0x00, 0x20)
            sleep(0.01)
            try:
                pasback_distance = car.pasback.read_distance()
                print("✔ pasback sensor read successfully.")
            except Exception as e:
                print(f"❌ Error reading pasback sensor: {e}")

            gpio.write(0x00, 0x40) 
            sleep(0.01) 
            try:
                drivback_distance = car.drivback.read_distance()
                print("✔ drivback sensor read successfully.")
            except Exception as e:
                print(f"❌ Error reading drivback sensor: {e}")

            gpio.write(0x00, 0x00)

            print("Sensor Readings:")
            print(f"  Driver:       {driver_distance}")
            print(f"  Passenger:    {passenger_distance}")
            print(f"  Front:        {front_distance}")
            print(f"  Pass Front:   {pasfront_distance}")
            print(f"  Drive Front:  {drifront_distance}")
            print(f"  Pass Back:    {pasback_distance}")
            print(f"  Drive Back:   {drivback_distance}")

            # Fuzzy logic
            turning_simulation.input['driver_sensor'] = passenger_distance
            turning_simulation.input['front_sensor'] = front_distance
            turning_simulation.input['passenger_sensor'] = driver_distance
            turning_simulation.input['rear_driver_sensor'] = drivback_distance
            turning_simulation.input['rear_passenger_sensor'] = pasback_distance
            turning_simulation.compute()

            pwm = turning_simulation.output.get('turning_degree')

            if pwm is not None:
                last_valid_pwm = pwm
                car.steering.set_pwm_time(50, pwm)
            else:
                print("⚠️  No valid steering output — using last good PWM value.")
                car.steering.set_pwm_time(50, last_valid_pwm)
                
except KeyboardInterrupt:
    car.steering.set_pwm_time(50, 800)
    car.motor.set_pwm_time(50, 1600)
    print("Stopped by user.")
    #WIRED

✔ pasback sensor read successfully.
✔ drivback sensor read successfully.
Sensor Readings:
  Driver:       7
  Passenger:    42
  Front:        25
  Pass Front:   23
  Drive Front:  21
  Pass Back:    142
  Drive Back:   25
✔ pasback sensor read successfully.
✔ drivback sensor read successfully.
Sensor Readings:
  Driver:       112
  Passenger:    42
  Front:        25
  Pass Front:   23
  Drive Front:  21
  Pass Back:    70
  Drive Back:   29
✔ pasback sensor read successfully.
✔ drivback sensor read successfully.
Sensor Readings:
  Driver:       35
  Passenger:    42
  Front:        25
  Pass Front:   23
  Drive Front:  27
  Pass Back:    187
  Drive Back:   26
✔ pasback sensor read successfully.
✔ drivback sensor read successfully.
Sensor Readings:
  Driver:       29
  Passenger:    19
  Front:        18
  Pass Front:   20
  Drive Front:  20
  Pass Back:    255
  Drive Back:   8
✔ pasback sensor read successfully.
✔ drivback sensor read successfully.
Sensor Readings:
  Driver:       

✔ pasback sensor read successfully.
✔ drivback sensor read successfully.
Sensor Readings:
  Driver:       160
  Passenger:    19
  Front:        25
  Pass Front:   55
  Drive Front:  13
  Pass Back:    52
  Drive Back:   7
✔ pasback sensor read successfully.
✔ drivback sensor read successfully.
Sensor Readings:
  Driver:       63
  Passenger:    19
  Front:        25
  Pass Front:   50
  Drive Front:  13
  Pass Back:    255
  Drive Back:   7
✔ pasback sensor read successfully.
✔ drivback sensor read successfully.
Sensor Readings:
  Driver:       15
  Passenger:    19
  Front:        24
  Pass Front:   55
  Drive Front:  13
  Pass Back:    255
  Drive Back:   7
✔ pasback sensor read successfully.
✔ drivback sensor read successfully.
Sensor Readings:
  Driver:       70
  Passenger:    19
  Front:        31
  Pass Front:   50
  Drive Front:  13
  Pass Back:    11
  Drive Back:   7
✔ pasback sensor read successfully.
✔ drivback sensor read successfully.
Sensor Readings:
  Driver:       138